# XGBoost Prediction

Pipeline: load → walk-forward CV (4 folds) → 2021 holdout

| Fold | Train end | Test window |
|------|-----------|-------------|
| 1 | 2019-12-30 | 2020-01-06 → 2020-03-30 |
| 2 | 2020-03-30 | 2020-04-06 → 2020-06-29 |
| 3 | 2020-06-29 | 2020-07-06 → 2020-09-28 |
| 4 | 2020-09-28 | 2020-10-05 → 2020-12-28 |
| Final | 2020-12-28 | 2021 full year |

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error


## 1. Load

In [2]:
df = pd.read_csv('../Feature_engineering/features_final.csv')
df['week_start'] = pd.to_datetime(df['week_start'])
df = df.sort_values(['week_start', 'product_id']).reset_index(drop=True)

TARGET   = 'sku_sold'
DROP_COLS = ['product_id', 'week_start', 'year', 'week', TARGET]
FEATURES  = [c for c in df.columns if c not in DROP_COLS]

print(f'Shape: {df.shape}')
print(f'Products: {df["product_id"].nunique()}')
print(f'Date range: {df["week_start"].min()} → {df["week_start"].max()}')
print(f'Features ({len(FEATURES)}): {FEATURES}')

Shape: (12700, 45)
Products: 115
Date range: 2018-02-19 00:00:00 → 2021-12-27 00:00:00
Features (40): ['lag_1', 'lag_2', 'lag_4', 'lag_8', 'rolling_mean_4', 'rolling_mean_8', 'rolling_std_4', 'rolling_std_8', 'month', 'week_of_year', 'is_month_start', 'is_month_end', 'month_sin', 'month_cos', 'week_sin', 'week_cos', 'has_holiday', 'min_days_to_holiday', 'hol_new_year', 'hol_easter', 'hol_kings_day', 'hol_liberation_day', 'hol_ascension', 'hol_pentecost', 'hol_christmas', 'temp_mean', 'temp_min', 'temp_max', 'precip_sum', 'sunshine_sum', 'temp_anomaly', 'heavy_rain', 'lockdown', 'lockdown_days', 'school_holiday', 'school_autumn', 'school_christmas', 'school_may', 'school_spring', 'school_summer']


## 2. Metrics

In [3]:
def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom == 0, 1e-8, denom)
    return np.mean(np.abs(y_true - y_pred) / denom) * 100

def wape(y_true, y_pred):
    return np.sum(np.abs(np.array(y_true) - np.array(y_pred))) / (np.sum(np.abs(y_true)) + 1e-8) * 100

def evaluate(y_true, y_pred):
    return {
        'MAE':   round(mean_absolute_error(y_true, y_pred), 4),
        'RMSE':  round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'SMAPE': round(smape(y_true, y_pred), 4),
        'WAPE':  round(wape(y_true, y_pred), 4),
    }

## 3. XGBoost

In [4]:
XGB_PARAMS = dict(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    random_state=42,
    verbosity=0,
)

## 4. Walk-Forward CV

In [5]:
FOLD_SPLITS = [
    ('2018-02-19', '2019-12-30', '2020-01-06', '2020-03-30'),
    ('2018-02-19', '2020-03-30', '2020-04-06', '2020-06-29'),
    ('2018-02-19', '2020-06-29', '2020-07-06', '2020-09-28'),
    ('2018-02-19', '2020-09-28', '2020-10-05', '2020-12-28'),
]

fold_results = []

for i, (tr_start, tr_end, te_start, te_end) in enumerate(FOLD_SPLITS):
    train = df[(df['week_start'] >= tr_start) & (df['week_start'] <= tr_end)]
    test  = df[(df['week_start'] >= te_start) & (df['week_start'] <= te_end)]

    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(train[FEATURES], train[TARGET].values)
    preds = np.clip(model.predict(test[FEATURES]), 0, None)

    metrics = evaluate(test[TARGET].values, preds)
    metrics.update({'fold': i + 1, 'train_rows': len(train),
                    'test_rows': len(test), 'train_end': tr_end,
                    'test_period': f'{te_start} → {te_end}'})
    fold_results.append(metrics)

    print(f"Fold {i+1} | {te_start} → {te_end} | "
          f"MAE={metrics['MAE']:.2f}  RMSE={metrics['RMSE']:.2f}  "
          f"SMAPE={metrics['SMAPE']:.2f}%  WAPE={metrics['WAPE']:.2f}%")

Fold 1 | 2020-01-06 → 2020-03-30 | MAE=26.95  RMSE=57.67  SMAPE=66.45%  WAPE=57.05%
Fold 2 | 2020-04-06 → 2020-06-29 | MAE=47.12  RMSE=89.99  SMAPE=61.38%  WAPE=44.86%
Fold 3 | 2020-07-06 → 2020-09-28 | MAE=20.60  RMSE=43.49  SMAPE=68.10%  WAPE=57.55%
Fold 4 | 2020-10-05 → 2020-12-28 | MAE=24.04  RMSE=49.84  SMAPE=61.73%  WAPE=44.44%


## 5. Final Holdout (2021)

In [6]:
train_final = df[df['week_start'] <= '2020-12-28']
test_final  = df[(df['week_start'] >= '2021-01-01') & (df['week_start'] <= '2021-12-31')]

final_model = xgb.XGBRegressor(**XGB_PARAMS)
final_model.fit(train_final[FEATURES], train_final[TARGET].values)
final_preds = np.clip(final_model.predict(test_final[FEATURES]), 0, None)

final_metrics = evaluate(test_final[TARGET].values, final_preds)
final_metrics.update({'fold': 'final_2021', 'train_rows': len(train_final),
                      'test_rows': len(test_final), 'train_end': '2020-12-28',
                      'test_period': '2021 full year'})
fold_results.append(final_metrics)

print(f"Final  | 2021 full year | "
      f"MAE={final_metrics['MAE']:.2f}  RMSE={final_metrics['RMSE']:.2f}  "
      f"SMAPE={final_metrics['SMAPE']:.2f}%  WAPE={final_metrics['WAPE']:.2f}%")

Final  | 2021 full year | MAE=30.87  RMSE=66.90  SMAPE=61.14%  WAPE=50.33%


## 6. CV Summary

In [7]:
results_df = pd.DataFrame(fold_results)
cv_folds   = results_df[results_df['fold'] != 'final_2021']

print('===== Walk-Forward Summary =====')
print(f"CV mean    — MAE: {cv_folds['MAE'].mean():.2f}  RMSE: {cv_folds['RMSE'].mean():.2f}  "
      f"SMAPE: {cv_folds['SMAPE'].mean():.2f}%  WAPE: {cv_folds['WAPE'].mean():.2f}%")
print(f"2021 final — MAE: {final_metrics['MAE']:.2f}  RMSE: {final_metrics['RMSE']:.2f}  "
      f"SMAPE: {final_metrics['SMAPE']:.2f}%  WAPE: {final_metrics['WAPE']:.2f}%")
print()
print(results_df.to_string(index=False))

===== Walk-Forward Summary =====
CV mean    — MAE: 29.68  RMSE: 60.25  SMAPE: 64.42%  WAPE: 50.97%
2021 final — MAE: 30.87  RMSE: 66.90  SMAPE: 61.14%  WAPE: 50.33%

    MAE    RMSE   SMAPE    WAPE       fold  train_rows  test_rows  train_end             test_period
26.9526 57.6679 66.4531 57.0487          1        6244        931 2019-12-30 2020-01-06 → 2020-03-30
47.1213 89.9929 61.3850 44.8555          2        7175        878 2020-03-30 2020-04-06 → 2020-06-29
20.6019 43.4930 68.1023 57.5502          3        8053        748 2020-06-29 2020-07-06 → 2020-09-28
24.0389 49.8366 61.7270 44.4403          4        8801        713 2020-09-28 2020-10-05 → 2020-12-28
30.8656 66.9024 61.1412 50.3324 final_2021        9514       3186 2020-12-28          2021 full year


## 7. Export

In [8]:
# Validation results
results_df.to_csv('xgb_wf_results.csv', index=False)

# Final predictions
pred_df = test_final[['product_id', 'week_start', TARGET]].copy()
pred_df['xgb_pred'] = final_preds
pred_df.to_csv('xgb_final_predictions.csv', index=False)